# CORNGUARD – Exploratory Data Analysis
**Dataset:** Corn / Maize Leaf Disease Dataset (Kaggle)  
**Classes:** Blight · Common Rust · Gray Leaf Spot · Healthy  
**Goal:** Understand the data distribution, image characteristics, and class imbalance before model training.


## 1. Imports

In [ ]:
import os
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
from PIL import Image
from collections import defaultdict

# Consistent style
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


## 2. Load Dataset Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/Acenas_Dataset/data'

CLASSES = sorted(os.listdir(DATA_DIR))
print("Classes found:", CLASSES)


## 3. Class Distribution

A balanced dataset allows a model to learn all classes equally well.  
We check here whether any class is significantly underrepresented.


In [ ]:
data = []
for cls in CLASSES:
    class_path = os.path.join(DATA_DIR, cls)
    for img_name in os.listdir(class_path):
        data.append({'path': os.path.join(class_path, img_name), 'label': cls})
df = pd.DataFrame(data)

class_counts = df['label'].value_counts().sort_index()
total = len(df)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
colors = ['#E36B6B', '#E8A838', '#6DAE8B', '#5B8EC4']
bars = axes[0].bar(class_counts.index, class_counts.values, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('Image Count per Class', fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Number of images')
for bar, count in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 str(count), ha='center', va='bottom', fontsize=10, fontweight='bold')

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    class_counts.values,
    labels=class_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=140,
    pctdistance=0.75,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for at in autotexts:
    at.set_fontsize(9)
axes[1].set_title('Class Proportion', fontweight='bold')

plt.suptitle('Class Distribution', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\nClass breakdown:")
for cls, cnt in class_counts.items():
    print(f"  {cls:<18} {cnt:>5} images  ({cnt/total*100:.1f}%).")
print(f"\n  Imbalance ratio (max/min): {class_counts.max()/class_counts.min():.2f}x")

## 4. Sample Images per Class

A visual check to confirm image quality and understand what each disease looks like.


In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
fig.suptitle('Sample Images per Class (5 per class)', fontsize=14, fontweight='bold')

for row, cls in enumerate(CLASSES):
    cls_df = df[df['label'] == cls]
    samples = cls_df.sample(5, random_state=SEED)['path'].tolist()
    for col, img_path in enumerate(samples):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(cls, fontsize=11, fontweight='bold',
                                       labelpad=8, rotation=90, va='center')

plt.tight_layout()
plt.show()


## 5. Image Dimension Analysis

Images in the wild come in varying sizes. We need to know the dominant dimensions  
so we can choose a sensible resize target that preserves detail without wasting memory.


In [ ]:
# Sample up to 100 images per class to keep this fast
dim_records = []
for cls in CLASSES:
    cls_df = df[df['label'] == cls]
    sample = cls_df.sample(min(100, len(cls_df)), random_state=SEED)
    for path in sample['path']:
        img = cv2.imread(path)
        if img is not None:
            h, w = img.shape[:2]
            dim_records.append({'label': cls, 'height': h, 'width': w,
                                 'aspect_ratio': round(w / h, 2)})

dim_df = pd.DataFrame(dim_records)

print("Overall dimension statistics:")
print(dim_df[['height', 'width']].describe().round(1))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Height distribution
for i, cls in enumerate(CLASSES):
    sub = dim_df[dim_df['label'] == cls]
    axes[0].hist(sub['height'], bins=20, alpha=0.55, label=cls, color=colors[i])
axes[0].set_title('Image Height Distribution')
axes[0].set_xlabel('Height (px)')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=8)

# Width distribution
for i, cls in enumerate(CLASSES):
    sub = dim_df[dim_df['label'] == cls]
    axes[1].hist(sub['width'], bins=20, alpha=0.55, label=cls, color=colors[i])
axes[1].set_title('Image Width Distribution')
axes[1].set_xlabel('Width (px)')
axes[1].set_ylabel('Count')
axes[1].legend(fontsize=8)

# Width vs Height scatter
for i, cls in enumerate(CLASSES):
    sub = dim_df[dim_df['label'] == cls]
    axes[2].scatter(sub['width'], sub['height'], alpha=0.4, s=20, label=cls, color=colors[i])
axes[2].axline((0,0), slope=1, color='gray', linestyle='--', linewidth=0.8, label='1:1 ratio')
axes[2].set_title('Width vs Height (per class)')
axes[2].set_xlabel('Width (px)')
axes[2].set_ylabel('Height (px)')
axes[2].legend(fontsize=8)

plt.suptitle('Image Dimension Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nAspect ratio stats per class:")
print(dim_df.groupby('label')['aspect_ratio'].describe()[['mean','min','max']].round(2))


## 6. Average Color Profile per Class

Different diseases leave distinct visual signatures.  
Comparing mean RGB channel values across classes shows whether color alone  
carries discriminative signal — useful context when choosing augmentation strategies.


In [ ]:
TARGET = (224, 224)   # resize for fast uniform comparison

channel_records = []
for cls in CLASSES:
    cls_df = df[df['label'] == cls]
    sample = cls_df.sample(min(80, len(cls_df)), random_state=SEED)
    means_r, means_g, means_b = [], [], []
    for path in sample['path']:
        img = cv2.imread(path)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, TARGET)
        means_r.append(img[:,:,0].mean())
        means_g.append(img[:,:,1].mean())
        means_b.append(img[:,:,2].mean())
    channel_records.append({
        'label': cls,
        'R': np.mean(means_r), 'G': np.mean(means_g), 'B': np.mean(means_b)
    })

ch_df = pd.DataFrame(channel_records)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Grouped bar: mean channel per class
x = np.arange(len(CLASSES))
w = 0.25
axes[0].bar(x - w, ch_df['R'], w, label='Red',   color='#E36B6B', edgecolor='white')
axes[0].bar(x,     ch_df['G'], w, label='Green', color='#6DAE8B', edgecolor='white')
axes[0].bar(x + w, ch_df['B'], w, label='Blue',  color='#5B8EC4', edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(CLASSES, rotation=15, ha='right')
axes[0].set_title('Mean RGB Channel per Class', fontweight='bold')
axes[0].set_ylabel('Mean pixel value (0–255)')
axes[0].legend()

# Average image mosaic – mean image per class
axes[1].axis('off')
axes[1].set_title('Mean Image per Class', fontweight='bold')

for idx, cls in enumerate(CLASSES):
    cls_df = df[df['label'] == cls]
    sample = cls_df.sample(min(60, len(cls_df)), random_state=SEED)
    stack = []
    for path in sample['path']:
        img = cv2.imread(path)
        if img is None: continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, TARGET)
        stack.append(img.astype(np.float32))
    mean_img = (np.mean(stack, axis=0) / 255.0).clip(0, 1)
    ax_ins = fig.add_axes([0.52 + (idx % 2) * 0.22, 0.55 - (idx // 2) * 0.45, 0.20, 0.38])
    ax_ins.imshow(mean_img)
    ax_ins.set_title(cls, fontsize=9, pad=3)
    ax_ins.axis('off')

plt.tight_layout()
plt.show()


## 7. Class Imbalance – Implication & Mitigation

`Gray_Leaf_Spot` has roughly **half** the images of the other classes.  
Without mitigation, the model will see it far less during training and may underperform on it.

**Mitigation strategy:** compute balanced class weights and pass them to `model.fit()`.  
This makes the loss penalize errors on underrepresented classes more heavily.


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

labels_array = df['label'].values
unique_classes = np.unique(labels_array)

# Encode labels to integers in alphabetical order (matches ImageDataGenerator)
label_to_idx = {cls: i for i, cls in enumerate(sorted(CLASSES))}
y_encoded = np.array([label_to_idx[l] for l in labels_array])

class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_encoded),
    y=y_encoded
)
class_weights = dict(enumerate(class_weights_arr))

print("Class weights (to pass into model.fit):")
for idx, cls in enumerate(sorted(CLASSES)):
    print(f"  [{idx}] {cls:<20} weight = {class_weights[idx]:.4f}")

# Visualise
fig, ax = plt.subplots(figsize=(7, 4))
weight_vals = [class_weights[i] for i in range(len(CLASSES))]
bars = ax.bar(sorted(CLASSES), weight_vals, color=colors, edgecolor='white')
ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8, label='Neutral weight = 1.0')
ax.set_title('Computed Class Weights (balanced)', fontweight='bold')
ax.set_ylabel('Weight')
ax.set_xlabel('Class')
for bar, w in zip(bars, weight_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{w:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


## 8. Pixel Intensity Distribution (Grayscale)

Shows whether images are well-exposed and whether classes differ in overall brightness —  
both useful for deciding whether normalization is needed and what augmentations make sense.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for i, cls in enumerate(CLASSES):
    cls_df = df[df['label'] == cls]
    sample = cls_df.sample(min(50, len(cls_df)), random_state=SEED)
    all_pixels = []
    for path in sample['path']:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, (128, 128))
            all_pixels.extend(img.flatten().tolist())
    ax.hist(all_pixels, bins=64, alpha=0.45, label=cls, color=colors[i], density=True)

ax.set_title('Grayscale Pixel Intensity Distribution per Class', fontweight='bold')
ax.set_xlabel('Pixel value (0 = black, 255 = white)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()


## 9. EDA Summary & Modeling Implications

| Finding | Implication |
|---|---|
| **4,188 total images** across 4 classes | Enough for fine-tuning a pretrained CNN |
| **Gray Leaf Spot is underrepresented** (574 imgs, ~2.3× fewer) | Class weights in the loss (below) |
| **Variable image sizes** (some > 3000 px) | Resize to a fixed 224×224 target |
| **Aspect ratios close to 1:1** for most images | Centre-crop to a square, then resize — the same thing the Android app does |
| **Colour profiles differ** across classes | Mild colour jitter only; strong hue shifts could erase disease colour cues |

**Preprocessing used for training (and by the CornGuard app):**
1. Centre-crop to a square, bilinear resize to 224×224
2. `mobilenet_v2.preprocess_input` → pixels scaled to **[-1, 1]** (`x / 127.5 − 1`).
   The app's `assets/model_config.json` must say `"normalization": "minus_one_to_one"`.
3. Balanced class weights
4. Training-only augmentation (flips, rotations, crops, colour jitter, JPEG compression)

# CORNGUARD – Model Training (v2, improved)
**Model:** MobileNetV2 (transfer learning, ImageNet weights) · **Classes:** Blight · Common Rust · Gray Leaf Spot · Healthy

### What changed from v1, and why
The v1 model reached 91.1% test accuracy, but it was weak on exactly the two lesion diseases:

| v1 test result | Blight | Common Rust | Gray Leaf Spot | Healthy |
|---|---|---|---|---|
| Recall | **0.78** | 0.97 | 0.86 | 1.00 |
| Precision | 0.92 | 0.96 | **0.69** | 0.98 |

Blight images were often predicted as Gray Leaf Spot (both are elongated tan lesions). v2 targets that:

1. **Train on the same pixels the app sees.** v1 resized with `flow_from_dataframe`, whose default is *nearest-neighbour* interpolation and a squash (no crop). The app centre-crops and resizes *bilinearly*. v2 uses a `tf.data` pipeline that does exactly what the app does.
2. **Remove duplicate images before splitting.** The Kaggle corn dataset contains duplicates; a copy in both train and test inflates the score, and a copy labelled with two different classes confuses training. v2 hashes every image, drops duplicates, and drops images whose duplicates disagree on the label.
3. **Field-photo augmentation.** Random square crops (zooming into lesions), 90° rotations + small tilts, flips, brightness/contrast/saturation/hue jitter and random JPEG compression — closer to phone photos taken in the field.
4. **Deeper, gentler fine-tuning.** Unfreeze from layer 60 (v1: 100), keep BatchNorm layers frozen, AdamW with warmup + cosine decay, and label smoothing 0.1 to reduce over-confident mistakes.
5. **Pick the best epoch by macro-F1** (every disease counts equally) instead of plain accuracy, which favours the large classes.
6. **Verify the exported TFLite model** on the test set with the app's preprocessing, and export the exact files the app needs.

> ⚙️ **Use a GPU runtime:** *Runtime → Change runtime type → T4 GPU*. v1 ran on CPU (`GPU available: []`, ~3–5 min per epoch).

In [ ]:
import os, random, json, shutil, itertools, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2, mobilenet_v2
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             roc_curve, auc)
from sklearn.preprocessing import label_binarize

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU available:", gpus)
if not gpus:
    print("⚠️  No GPU — training will be very slow. Runtime → Change runtime type → T4 GPU.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA_DIR = '/content/drive/MyDrive/Acenas_Dataset/data'
MODEL_DIR      = '/content/drive/MyDrive/Acenas_Dataset/models'
os.makedirs(MODEL_DIR, exist_ok=True)

# Copy the dataset to the Colab VM once: reading thousands of small files from Drive every
# epoch is the main bottleneck.
DATA_DIR = '/content/data'
if not os.path.isdir(DATA_DIR):
    shutil.copytree(DRIVE_DATA_DIR, DATA_DIR)

IMAGE_SIZE      = (224, 224)   # must match "input_size" in the app's model_config.json
BATCH_SIZE      = 32
EPOCHS_HEAD     = 12           # phase 1: classification head only
EPOCHS_FINE     = 30           # phase 2: fine-tune (early stopping usually ends it sooner)
LR_HEAD         = 1e-3
LR_FINE         = 3e-5         # peak LR after warmup
UNFREEZE_FROM   = 60           # v1 used 100; unfreezing more lets the base learn lesion shapes
LABEL_SMOOTHING = 0.1
MODEL_VERSION   = 'cornguard_mobilenetv2_v3'

CLASSES = sorted(os.listdir(DATA_DIR))
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
print("Classes:", CLASSES)

## Build image table & remove duplicates
Perceptual hashes (pHash) are identical for exact copies and for re-saved / resized copies of the same photo.
* same hash, same class → keep one copy
* same hash, **different** classes → drop all copies (the label is unreliable)

In [ ]:
!pip -q install imagehash
import imagehash

records = []
for cls in CLASSES:
    for fname in os.listdir(os.path.join(DATA_DIR, cls)):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            records.append({'path': os.path.join(DATA_DIR, cls, fname), 'label': cls})
df = pd.DataFrame(records)
print("Images found:", len(df))

def phash(path):
    try:
        with Image.open(path) as im:
            return str(imagehash.phash(im.convert('RGB')))
    except Exception:
        return None

df['hash'] = [phash(p) for p in tqdm(df['path'], desc='Hashing')]
unreadable = df['hash'].isna().sum()
df = df.dropna(subset=['hash'])

labels_per_hash = df.groupby('hash')['label'].nunique()
conflicting = set(labels_per_hash[labels_per_hash > 1].index)
n_conflict = df['hash'].isin(conflicting).sum()
df = df[~df['hash'].isin(conflicting)]
n_before = len(df)
df = df.drop_duplicates(subset='hash', keep='first')

print(f"Unreadable files dropped:            {unreadable}")
print(f"Images with conflicting labels dropped: {n_conflict}")
print(f"Duplicate copies dropped:            {n_before - len(df)}")
print(f"Images kept:                         {len(df)}\n")
print(df['label'].value_counts().sort_index())

In [ ]:
# Train / Val / Test split (70 / 15 / 15), stratified, after de-duplication (no leakage)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=SEED)
val_df, test_df   = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

y_train = train_df['label'].map(CLASS_TO_IDX).values
cw = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train)
class_weights = dict(enumerate(cw))
for i, c in enumerate(CLASSES):
    print(f"  [{i}] {c:<16} weight {class_weights[i]:.3f}")

## Input pipeline (same preprocessing as the app)
* **Eval / app:** centre-crop to a square → bilinear resize to 224×224 → `preprocess_input` ([-1, 1]).
* **Train:** random square crop (75–100% of the short side) → bilinear resize → flips, 90° rotations, colour jitter, JPEG compression → `preprocess_input` → small random tilt.

Class weights are applied as per-sample weights, which works the same on every Keras version.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def decode(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    return tf.cast(img, tf.float32) / 255.0           # [0, 1] for the colour ops

def center_square(img):
    h, w = tf.shape(img)[0], tf.shape(img)[1]
    s = tf.minimum(h, w)
    return tf.image.crop_to_bounding_box(img, (h - s) // 2, (w - s) // 2, s, s)

def eval_preprocess(img):
    return tf.image.resize(center_square(img), IMAGE_SIZE, method='bilinear')

def train_preprocess(img):
    s = tf.minimum(tf.shape(img)[0], tf.shape(img)[1])
    crop = tf.cast(tf.cast(s, tf.float32) * tf.random.uniform([], 0.75, 1.0), tf.int32)
    img = tf.image.random_crop(img, tf.stack([crop, crop, 3]))
    img = tf.image.resize(img, IMAGE_SIZE, method='bilinear')
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, tf.random.uniform([], 0, 4, dtype=tf.int32))
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.image.random_saturation(img, 0.8, 1.2)
    img = tf.image.random_hue(img, 0.03)             # small: colour is a disease cue
    img = tf.clip_by_value(img, 0.0, 1.0)
    img = tf.image.random_jpeg_quality(img, 60, 100) # phone-camera compression
    return img

def to_model_input(img01):
    return mobilenet_v2.preprocess_input(img01 * 255.0)   # -> [-1, 1]

random_tilt = layers.RandomRotation(0.05, fill_mode='reflect', seed=SEED)

def make_dataset(frame, training):
    paths = frame['path'].values
    labels = frame['label'].map(CLASS_TO_IDX).values.astype(np.int32)
    if training:
        weights = np.array([class_weights[l] for l in labels], dtype=np.float32)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels, weights))
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(lambda p, l, w: (to_model_input(train_preprocess(decode(p))),
                                     tf.one_hot(l, NUM_CLASSES), w),
                    num_parallel_calls=AUTOTUNE)
        ds = ds.batch(BATCH_SIZE)
        ds = ds.map(lambda x, y, w: (random_tilt(x, training=True), y, w), num_parallel_calls=AUTOTUNE)
    else:
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        ds = ds.map(lambda p, l: (to_model_input(eval_preprocess(decode(p))), tf.one_hot(l, NUM_CLASSES)),
                    num_parallel_calls=AUTOTUNE)
        ds = ds.batch(BATCH_SIZE).cache()
    return ds.prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds   = make_dataset(val_df, training=False)
test_ds  = make_dataset(test_df, training=False)
y_val  = val_df['label'].map(CLASS_TO_IDX).values
y_true = test_df['label'].map(CLASS_TO_IDX).values
steps_per_epoch = int(tf.data.experimental.cardinality(train_ds))
print("Steps per epoch:", steps_per_epoch)

In [ ]:
# Visual check of the training augmentation
xb, yb, _ = next(iter(train_ds))
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax, x, y in zip(axes.flatten(), xb[:12], yb[:12]):
    ax.imshow(((x.numpy() + 1) / 2).clip(0, 1))
    ax.set_title(IDX_TO_CLASS[int(np.argmax(y))], fontsize=9)
    ax.axis('off')
plt.suptitle('Augmented training samples', fontweight='bold')
plt.tight_layout(); plt.show()

## Model & macro-F1 callback

In [ ]:
def build_model(num_classes):
    base = MobileNetV2(input_shape=(*IMAGE_SIZE, 3), include_top=False, weights='imagenet')
    base.trainable = False
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
    x = base(inputs, training=False)          # BatchNorm stays in inference mode
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.40)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return Model(inputs, outputs), base

class MacroF1(tf.keras.callbacks.Callback):
    # Adds val_macro_f1 to the epoch logs so checkpoints/early stopping can use it.
    def __init__(self, ds, y):
        super().__init__()
        self.ds, self.y = ds, y
    def on_epoch_end(self, epoch, logs=None):
        pred = self.model.predict(self.ds, verbose=0).argmax(axis=1)
        logs['val_macro_f1'] = f1_score(self.y, pred, average='macro')
        print(f" — val_macro_f1: {logs['val_macro_f1']:.4f}")

model, base = build_model(NUM_CLASSES)
loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)
model.summary()

## Phase 1 – Train the classification head

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(LR_HEAD), loss=loss, metrics=['accuracy'])

head_path = os.path.join(MODEL_DIR, 'cornguard_v3_head_best.keras')
history_head = model.fit(
    train_ds, epochs=EPOCHS_HEAD, validation_data=val_ds,
    callbacks=[
        MacroF1(val_ds, y_val),     # must come before the callbacks that read val_macro_f1
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max', patience=4,
                                         restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(head_path, monitor='val_macro_f1', mode='max',
                                           save_best_only=True, verbose=1),
    ],
)

## Phase 2 – Fine-tune
Unfreeze MobileNetV2 from layer `UNFREEZE_FROM`, but keep **BatchNorm** layers frozen (their running
statistics come from ImageNet and are easily damaged by small batches). The learning rate warms up
for 2 epochs, then follows a cosine decay; AdamW adds mild weight decay.

In [ ]:
base.trainable = True
for layer in base.layers[:UNFREEZE_FROM]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
print(f"Trainable layers in MobileNetV2: {sum(l.trainable for l in base.layers)} / {len(base.layers)}")

schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=LR_FINE / 10,
    decay_steps=steps_per_epoch * EPOCHS_FINE,
    alpha=0.05,
    warmup_target=LR_FINE,
    warmup_steps=steps_per_epoch * 2,
)
model.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=schedule, weight_decay=1e-4),
              loss=loss, metrics=['accuracy'])

fine_path = os.path.join(MODEL_DIR, 'cornguard_v3_finetuned_best.keras')
history_fine = model.fit(
    train_ds, epochs=EPOCHS_FINE, validation_data=val_ds,
    callbacks=[
        MacroF1(val_ds, y_val),
        tf.keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max', patience=8,
                                         restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(fine_path, monitor='val_macro_f1', mode='max',
                                           save_best_only=True, verbose=1),
    ],
)
best_model = model   # best weights restored by EarlyStopping

## Training history

In [ ]:
def plot_history(h1, h2, metric):
    tr = h1.history[metric] + h2.history[metric]
    va = h1.history[f'val_{metric}'] + h2.history[f'val_{metric}']
    ep = range(1, len(tr) + 1)
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(ep, tr, label=f'Train {metric}', color='#5B8EC4')
    ax.plot(ep, va, label=f'Val {metric}', color='#E36B6B')
    ax.axvline(len(h1.history[metric]), color='gray', linestyle='--', linewidth=1, label='Fine-tune starts')
    ax.set_title(f'Training vs Validation {metric}', fontweight='bold'); ax.set_xlabel('Epoch'); ax.legend()
    plt.tight_layout(); plt.show()

plot_history(history_head, history_fine, 'accuracy')
plot_history(history_head, history_fine, 'loss')

f1_curve = history_head.history['val_macro_f1'] + history_fine.history['val_macro_f1']
plt.figure(figsize=(11, 3)); plt.plot(range(1, len(f1_curve) + 1), f1_curve, color='#6DAE8B')
plt.title('Validation macro-F1', fontweight='bold'); plt.xlabel('Epoch'); plt.tight_layout(); plt.show()

## Evaluation on the test set

In [ ]:
y_pred_proba = best_model.predict(test_ds, verbose=1)
y_pred = y_pred_proba.argmax(axis=1)
class_names = [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]

test_acc = (y_pred == y_true).mean()
test_macro_f1 = f1_score(y_true, y_pred, average='macro')
print(f"Test accuracy : {test_acc*100:.2f}%   (v1: 91.10%)")
print(f"Test macro-F1 : {test_macro_f1:.4f}   (v1: 0.8914)\n")
report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)

In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, data, title, fmt in [(axes[0], cm, 'Counts', 'd'), (axes[1], cm_norm, 'Row-normalised (recall)', '.2f')]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues', xticklabels=class_names,
                yticklabels=class_names, ax=ax, cbar=False)
    ax.set_title(f'Confusion Matrix – {title}', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout(); plt.show()

print("Most common confusions:")
pairs = [(cm[i, j], class_names[i], class_names[j]) for i in range(NUM_CLASSES)
         for j in range(NUM_CLASSES) if i != j and cm[i, j] > 0]
for n, t, p in sorted(pairs, reverse=True)[:6]:
    print(f"  {t:<15} → predicted {p:<15} {n} images")

In [ ]:
# Error analysis: look at what the model gets wrong
wrong = np.where(y_pred != y_true)[0]
print(f"{len(wrong)} misclassified test images")
show = wrong[:16]
if len(show):
    cols = 4; rows = int(np.ceil(len(show) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    for ax in np.array(axes).flatten(): ax.axis('off')
    for ax, i in zip(np.array(axes).flatten(), show):
        ax.imshow(Image.open(test_df.iloc[i]['path']).convert('RGB'))
        ax.set_title(f"True: {class_names[y_true[i]]}\nPred: {class_names[y_pred[i]]} "
                     f"({y_pred_proba[i].max()*100:.0f}%)", fontsize=9, color='#c62828')
    plt.tight_layout(); plt.show()

In [ ]:
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(8, 6))
for i, (cls, color) in enumerate(zip(class_names, ['#E36B6B', '#E8A838', '#6DAE8B', '#5B8EC4'])):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{cls} (AUC = {auc(fpr, tpr):.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
ax.set_title('ROC-AUC Curves – Test Set', fontweight='bold')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate'); ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

## Export to TFLite and verify it
The TFLite model (dynamic-range quantised, like v1) is re-evaluated on the test set with the **same preprocessing the app uses**, so the accuracy you see here is what the app gets.

In [ ]:
final_path = os.path.join(MODEL_DIR, 'cornguard_v3_final.keras')
best_model.save(final_path)
print("Saved:", final_path)

converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
tflite_path = os.path.join(MODEL_DIR, 'cornguard_v3_model.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
print(f"TFLite model: {tflite_path} ({os.path.getsize(tflite_path)/1e6:.2f} MB)")

interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]
tfl_pred = []
for x, _ in test_ds.unbatch():
    interpreter.set_tensor(inp['index'], x.numpy()[None].astype(np.float32))
    interpreter.invoke()
    tfl_pred.append(interpreter.get_tensor(out['index'])[0].argmax())
tfl_pred = np.array(tfl_pred)
print(f"TFLite test accuracy : {(tfl_pred == y_true).mean()*100:.2f}%")
print(f"TFLite test macro-F1 : {f1_score(y_true, tfl_pred, average='macro'):.4f}")
print(f"Agreement with Keras : {(tfl_pred == y_pred).mean()*100:.2f}%")

## Export files for the CornGuard Android app
Creates `MODEL_DIR/android_assets/` with the four files the app reads. Copy them into
`CornGuard/android/app/src/main/assets/` (replacing the old ones) and rebuild the app.

The model's "Blight" class is shown in the app as **Northern Leaf Blight** (`northern_leaf_blight`).

In [ ]:
APP_LABELS = {
    'Blight':         ('northern_leaf_blight', 'Northern Leaf Blight'),
    'Common_Rust':    ('common_rust',          'Common Rust'),
    'Gray_Leaf_Spot': ('gray_leaf_spot',       'Gray Leaf Spot'),
    'Healthy':        ('healthy',              'Healthy'),
}
assets_dir = os.path.join(MODEL_DIR, 'android_assets')
os.makedirs(assets_dir, exist_ok=True)

shutil.copy(tflite_path, os.path.join(assets_dir, 'model.tflite'))
labels = [{'index': i, 'disease_code': APP_LABELS[c][0], 'display_label': APP_LABELS[c][1]}
          for i, c in enumerate(CLASSES)]
with open(os.path.join(assets_dir, 'labels.json'), 'w') as f:
    json.dump(labels, f, indent=2)
with open(os.path.join(assets_dir, 'model_version.txt'), 'w') as f:
    f.write(MODEL_VERSION)
with open(os.path.join(assets_dir, 'model_config.json'), 'w') as f:
    json.dump({'input_size': IMAGE_SIZE[0], 'normalization': 'minus_one_to_one'}, f, indent=2)

# caps 3-style label map, as in v1
with open(os.path.join(MODEL_DIR, 'labels.json'), 'w') as f:
    json.dump({str(i): c for i, c in enumerate(CLASSES)}, f, indent=2)

print("App assets written to", assets_dir)
for name in sorted(os.listdir(assets_dir)):
    print("  ", name)
print(json.dumps(labels, indent=2))

## (Optional) Test on real field photos
Lab-style dataset accuracy is usually higher than accuracy on phone photos from the field. If you have
field photos, put them in Drive as `Acenas_Dataset/field_test/<ClassName>/*.jpg` (same class folder names
as the dataset) and run this cell — it uses the TFLite model with the app's preprocessing.

In [ ]:
FIELD_DIR = '/content/drive/MyDrive/Acenas_Dataset/field_test'
if os.path.isdir(FIELD_DIR):
    rows = [{'path': os.path.join(FIELD_DIR, c, f), 'label': c}
            for c in CLASSES if os.path.isdir(os.path.join(FIELD_DIR, c))
            for f in os.listdir(os.path.join(FIELD_DIR, c)) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    field_df = pd.DataFrame(rows)
    field_true = field_df['label'].map(CLASS_TO_IDX).values
    field_pred, field_conf = [], []
    for x, _ in make_dataset(field_df, training=False).unbatch():
        interpreter.set_tensor(inp['index'], x.numpy()[None].astype(np.float32))
        interpreter.invoke()
        probs = interpreter.get_tensor(out['index'])[0]
        field_pred.append(probs.argmax()); field_conf.append(probs.max())
    field_df['predicted'] = [IDX_TO_CLASS[i] for i in field_pred]
    field_df['confidence'] = np.round(np.array(field_conf) * 100, 1)
    field_df['correct'] = field_df['predicted'] == field_df['label']
    print(f"Field accuracy: {field_df['correct'].mean()*100:.1f}% on {len(field_df)} photos")
    display(field_df[['path', 'label', 'predicted', 'confidence', 'correct']])
else:
    print("No field_test folder — skipped.")

In [ ]:
print("Final Model Performance (Test Set)")
print("-" * 45)
print(f"Keras  accuracy : {test_acc*100:.2f}%   macro-F1 {test_macro_f1:.4f}")
print(f"TFLite accuracy : {(tfl_pred == y_true).mean()*100:.2f}%   macro-F1 {f1_score(y_true, tfl_pred, average='macro'):.4f}")
print("\n" + report)